# Lecture 19 - Architecture and data stores

**Lecture Topic**: The shape of a whole data system. How the pieces fit together, where the data lives, and the trade-offs you are choosing between. First lecture of Module D.

## Announcements

- **Module C is done.** You can now reason about how a piece of code scales.
- **Module D starts today**: software engineering. Architecture, structuring code, and testing.
- **M2 (design) is due Monday, Nov 23.** Today's material will help you describe your architecture and scaling analysis.
- ClassBuzz poll for today: `XXXX`

## Where we are

- **Module A**: the data tools.
- **Module B**: working with a coding agent.
- **Module C**: predicting how code scales.
- **Module D**: building something that lasts. Architecture, structure, and testing.

Module C was about a single computation. Today is about the *system* around it: the stages, how data moves between them, and where it is stored.

Today:
 - What architecture means, and the paradigms for organising code
 - Pipelines, layered design, and a first look at design patterns
 - The same calculation built three ways
 - Data stores and file formats
 - The trade-off you are always making

# What architecture means here

Architecture is the shape of the whole system: the stages, how data flows between them, and where each piece lives. It is not about one function, it is about how the parts fit and what you are trading off between them.

You already met the core idea last lecture. A data pipeline is a **DAG** of stages, and architecture is the work of designing that DAG and choosing where the data sits at each step.

An agent will happily write you a single script that does everything, because that is the shortest path to a working answer. Knowing some architecture is what lets you ask for the *right* shape instead, and judge the shape it gives you.

There are really three zoom levels to "how is this organised", and we will touch all three over this lecture and the next:

 - **Architecture**: the shape of the whole system (the stages, the flow, the storage). Today.
 - **Paradigm**: how you organise the code inside it (procedural, object-oriented, functional, declarative). Next, below.
 - **Pattern**: a named solution to a recurring design problem (strategy, factory, and friends). A taste today, properly next lecture.

# Ways to organise the code: paradigms

Zoom in one level from the system. The next choice is the *style* in which you write the code, and there are a few broad ones. Real code mixes them.

 - **Procedural**: a sequence of steps and functions acting on data. The default for a script.
 - **Object-oriented**: bundle data together with the behaviour that acts on it, and model the problem as objects. Good when there is state to manage.
 - **Functional**: build the result by composing transformations, keeping functions pure and avoiding hidden mutable state. The natural fit for transforming data.
 - **Declarative**: say *what* you want and let the engine work out *how*. SQL, a regular expression, a pandas query, xarray's label-based operations.

## Which one, and when

Data-science code leans **functional**: you chain transformations over a dataframe and let pandas do the work. You reach for **objects** when something has real state (a model you fit and then predict with, a connection you open and close), and you drop into **declarative** at the edges (a SQL query, a regex). The skill is matching the style to the piece.

 - Functional code is easy to test and to parallelise, because a pure function has no hidden state. It can be awkward for things that are genuinely stateful.
 - Objects model state well, and tempt you to wrap everything in a class that never needed one.
 - Procedural is the simplest, and tangles as it grows.
 - Declarative is concise and the engine can optimise it, at the cost of some control.

## The same job, three ways

One calculation, mean tip per zone, written in three paradigms. Watch what changes.

In [ ]:
import pandas as pd

taxi = pd.read_csv("../data/2019_Yellow_Taxi_Trip_Data.csv")

# procedural: step through the zones, building up a result
def mean_tip_procedural(df):
    result = {}
    for zone in df["pulocationid"].unique():
        sub = df[df["pulocationid"] == zone]          # a scan per zone: O(n*k)
        result[zone] = sub["tip_amount"].mean()
    return result

# object-oriented: bundle the data together with the behaviour
class ZoneTips:
    def __init__(self, df):
        self.df = df
    def mean_tip(self):
        return self.df.groupby("pulocationid")["tip_amount"].mean()

# functional: compose transformations, no loop and no state
def mean_tip_functional(df):
    return df.groupby("pulocationid")["tip_amount"].mean()

functional = mean_tip_functional(taxi)
oop = ZoneTips(taxi).mean_tip()
z = functional.index[0]
print("functional == object-oriented:", functional.equals(oop))
print("procedural agrees on zone", z, ":",
      abs(mean_tip_procedural(taxi)[z] - functional.loc[z]) < 1e-9)

All three give the same numbers, and the paradigm you pick changes more than the look.

 - The **functional** one-liner is the clearest and the fastest, because `groupby` is a single O(n) pass.
 - The **procedural** loop is the quadratic from Module C, a scan per zone.
 - The **object** wraps one function in state it does not have, which is the over-use of OOP we keep warning about.

Pick the style that fits the piece, and most data work fits functional.

# Pipelines and layered design

A **pipeline** breaks the work into stages, each with a clear input and output:

```
raw data  ->  load  ->  clean  ->  features  ->  analysis  ->  report
```

A **layered** design does the same for storage, keeping the data at a few named stages:

 - **raw**: exactly what you received, never edited.
 - **cleaned**: validated and typed, the version everything downstream reads.
 - **curated**: the finished tables and figures.

Why bother, when one script would run end to end? Because stages you can name are stages you can:

 - **cache**: skip a step whose inputs have not changed (the idea behind `make` and Dask from last lecture).
 - **test**: check each stage's output on its own.
 - **re-run**: restart from the cleaned layer after a failure, instead of from the raw download.
 - **reason about**: put a cost and a scaling estimate on each stage (Module C), rather than on one tangled block.

Keeping the raw layer untouched also means a bug in cleaning is always recoverable. You never overwrite the only copy.

# Design patterns: named solutions

Zoom in one more level. Inside a paradigm, the same design problems come up again and again, and the good solutions have names. A **design pattern** is one of those named, reusable solutions.

The **pipeline** you just drew is one of them: a named way to structure a flow as independent stages. There is a small vocabulary of these (strategy, factory, adapter, decorator, and a handful more), and two reasons they are worth knowing.

 - They are a **compression of experience**. A pattern packages a solution other people have already debugged, so you are not reinventing it from scratch.
 - They are a **shared vocabulary with your agent**. The patterns are all over its training data, so naming one is a precise instruction. "Make the loader a factory" or "wrap it in a decorator" hands over a whole design in a few words.

So architecture, paradigm, and pattern are three zoom levels of one question: how do you organise the code so that you, and your agent, can reason about it? We go through the patterns properly next lecture. Today, just notice that the pipeline is the first one.

# The same calculation, three ways

Take one real calculation: the area-weighted global mean temperature from the climate data (the one from the xarray lecture). The *logic* is fixed. The *architecture* around it is a choice, and it depends on who uses it and how often.

## Shape 1: a one-off analysis

A few lines in a notebook. Perfect for looking once.

In [ ]:
import numpy as np
import xarray as xr

ds = xr.open_dataset("../data/global_temparature.nc")
weights = np.cos(np.deg2rad(ds["lat"]))
global_mean = ds["tas"].weighted(weights).mean(["lat", "lon"])
 
print("global mean, year 2000:", round(float(global_mean.sel(time=2000)), 2))

Quick to write and easy to read. But there is nothing to re-run tomorrow except by hand, no test, and nothing anyone else can call. That is fine for a question you ask once.

## Shape 2: a nightly pipeline

The same logic as named stages, with a cached intermediate so a re-run skips work that is already done. This is the shape for something that runs on a schedule as new data arrives.

In [ ]:
import os
import tempfile

def load(path):
    return xr.open_dataset(path)

def compute(ds):
    w = np.cos(np.deg2rad(ds["lat"]))
    return ds["tas"].weighted(w).mean(["lat", "lon"])

def run_pipeline(src, cache):
    if os.path.exists(cache):
        print("  cache hit: skipping recompute")
        return xr.open_dataarray(cache)
    print("  computing and caching")
    result = compute(load(src))
    result.to_netcdf(cache)
    return result

cache = os.path.join(tempfile.gettempdir(), "global_mean.nc")
if os.path.exists(cache):
    os.remove(cache)

print("first run:");  run_pipeline("../data/global_temparature.nc", cache)
print("second run:"); run_pipeline("../data/global_temparature.nc", cache)

The second run is instant because the stage is **idempotent**: given the same input it produces the same output, so it can safely be skipped. That is what lets a nightly job re-run cheaply and restart after a failure. It costs more to set up than the notebook.

## Shape 3: a shared package

The same logic as a clean function others import, with a docstring and a test it ships with. This is the shape for something a team or a future project will depend on.

In [ ]:
def global_weighted_mean(ds, var="tas"):
    """Area-weighted global mean of `var` over lat/lon, one value per time step."""
    weights = np.cos(np.deg2rad(ds["lat"]))
    return ds[var].weighted(weights).mean(["lat", "lon"])

def test_global_weighted_mean():
    ds = xr.open_dataset("../data/global_temparature.nc")
    out = global_weighted_mean(ds)
    assert out.dims == ("time",)
    assert out.sizes["time"] == 250

test_global_weighted_mean()
print("packaged function and its test: OK")

Most work of the three, and worth it only when others depend on it. A clear interface, a docstring, and a test are what make it safe for someone else to use without reading every line.

## Same logic, three architectures

| | One-off | Pipeline | Package |
|---|---|---|---|
| Effort | minutes | hours | days |
| Re-runnable | by hand | scheduled, cached | yes, by anyone |
| Tested | no | per stage | yes |
| Who it's for | you, once | a scheduled job | other people / projects |
| Reach for it when | exploring | it runs regularly on new data | others will build on it |

The judgment is matching the shape to the need. An agent defaults to Shape 1, because it is the shortest path to an answer. Over-building a one-off into a package wastes a day; under-building a production job as a notebook breaks the first time the data changes. You choose the shape; the agent builds it.

### Question 1

You need the global-mean number once, to put on a slide for a meeting tomorrow. Which shape should you ask the agent for?

 - A. A shared, tested package
 - B. A scheduled nightly pipeline
 - C. A few lines in a notebook
 - D. A database with an API

**C.** For a number you need once, the notebook is the right amount of engineering. The skill is resisting the urge to build Shape 3 for a Shape 1 problem, and knowing when you genuinely do need the pipeline or the package.

# Data stores: where the data lives

The other half of architecture is storage. The format you pick is a performance decision, the same cost-and-scaling thinking from Module C applied to bytes on disk.

The one most agents reach for by default is CSV, and for a large table it is usually the wrong choice.

## CSV vs. Parquet

**CSV** is text, one row per line. It is universal and human-readable, but it is row-oriented and untyped: every value is a string until you parse it, and reading one column still means reading every row.

**Parquet** is a binary, **columnar**, compressed, typed format. It stores each column together, so you can read just the columns you want, and it is much smaller. Let us measure it.

In [ ]:
import pandas as pd
import os
import tempfile

taxi = pd.read_csv("../data/2019_Yellow_Taxi_Trip_Data.csv")
big = pd.concat([taxi] * 50, ignore_index=True)        # ~500k rows, a more realistic size

tmp = tempfile.gettempdir()
csv_path = os.path.join(tmp, "taxi_big.csv")
pq_path = os.path.join(tmp, "taxi_big.parquet")
big.to_csv(csv_path, index=False)
big.to_parquet(pq_path)

print(f"rows:    {len(big):,}")
print(f"CSV      {os.path.getsize(csv_path) / 1e6:5.1f} MB")
print(f"Parquet  {os.path.getsize(pq_path) / 1e6:5.1f} MB")

In [ ]:
import time

def ms(fn):
    start = time.perf_counter()
    fn()
    return (time.perf_counter() - start) * 1000

print("read every column:")
print(f"  CSV      {ms(lambda: pd.read_csv(csv_path)):5.0f} ms")
print(f"  Parquet  {ms(lambda: pd.read_parquet(pq_path)):5.0f} ms")
print("read one column (tip_amount):")
print(f"  CSV      {ms(lambda: pd.read_csv(csv_path, usecols=['tip_amount'])):5.0f} ms")
print(f"  Parquet  {ms(lambda: pd.read_parquet(pq_path, columns=['tip_amount'])):5.0f} ms")

Parquet is many times smaller and, because it is columnar, reading one column barely touches the file while CSV has to parse every row to find it. The gap grows with the size of the data and with how few columns you need. For anything you will read more than once, writing a Parquet copy pays for itself quickly.

### Question 2

You have a 50 GB table of 40 columns, and your analysis repeatedly reads the same 3 of them. Which store will serve those reads best?

 - A. One big CSV file
 - B. A Parquet file
 - C. A folder of JSON files, one per row
 - D. A single Python pickle

**B, Parquet.** Columnar storage reads only the 3 columns you ask for and skips the other 37 on disk, and the compression keeps the file small. CSV and JSON would parse all 40 columns every time; a pickle would load the whole 50 GB into memory at once and is not portable or safe to share.

## The rest of the shelf

You will meet a few more, each suited to a shape of data:

| Store | Good at | Reach for it when |
|---|---|---|
| **CSV** | universal, human-readable | small data, or handing a file to a human |
| **Parquet** | columnar, compressed, typed | tabular analytics at any real size |
| **NetCDF / Zarr** | labelled N-dimensional arrays | gridded science data (Zarr for cloud and parallel reads) |
| **SQLite / Postgres** | indexed lookups, updates, concurrency | many small queries, or several writers at once |

The pattern: columnar formats (Parquet, Zarr) win when you scan and aggregate; databases win when you look things up and change them. NetCDF and Zarr are the array version of the same idea, and the ones behind the xarray data from Module A.

# The trade-off you are always making

Every architecture choice trades off the same four things, and it is worth naming them stage by stage:

 - **Cost**: compute, storage, and your time to build and maintain it.
 - **Scalability**: does it still work at ten times the data?
 - **Reliability**: what happens when a stage fails or the data is malformed?
 - **Speed**: is it fast enough for how it is used?

The three shapes trade effort for reuse and reliability. The storage formats trade simplicity for speed and scale. When an agent hands you a design, walk its stages and ask these four questions about each one.

# Architecture is a vocabulary

This connects straight back to Module B. An agent will build whatever shape you ask for, so the more precisely you can name the shape, the better the result.

"Write a script that computes the global mean" gets you Shape 1. "Build this as a staged pipeline with a cached Parquet intermediate per stage, idempotent so a re-run skips finished stages" gets you Shape 2, in one sentence. The words *pipeline*, *idempotent*, *columnar*, and *Parquet* each carry a whole design that the agent already understands. The paradigm and the pattern are vocabulary too: "write this as pure functions" or "make the loader a factory" steer it just as compactly.

So the job is not to write the architecture by hand. It is to know enough of the vocabulary to ask for the right one, and to judge what comes back: does the shape match who uses it and how often, and does each stage hold up on cost, scalability, reliability, and speed?

That judgment is exactly what M2 is asking you to show.

# Recap

 - **Architecture** is the shape of the whole system: the stages, the flow, and where data lives. Zoom in and you choose a **paradigm** (functional for most data work, objects for state) and reach for a named **pattern** when one fits.
 - A **pipeline** is a DAG of stages and the first design pattern you meet; a **layered** design keeps raw, cleaned, and curated data so you can cache, test, re-run, and reason per stage.
 - The same logic fits a **one-off**, a **pipeline**, or a **package**; match the shape to who uses it and how often.
 - **Storage is a performance choice**: Parquet (columnar) for tabular analytics, Zarr/NetCDF for arrays, a database for lookups and writes. CSV is the easy default and usually the wrong one at scale.
 - Judge any design on **cost, scalability, reliability, and speed**, stage by stage, and name the shape you want when you ask an agent for it.

**Next time (S22): structuring code** - from a notebook to a package, and the design patterns that are a shared vocabulary for you and your agent.